# 03 — Aggregator — the shared vocabulary

| | |
|---|---|
| **runs at** | the aggregator |
| **reads** | `cohort-A_features.csv`, `cohort-B_features.csv` |
| **writes** | `shared_features.csv` |

The first aggregator step, and the smallest. It receives two **feature lists** and
nothing else: no measurements, no samples, no counts.

**Why this step has to exist.** A cluster is identified by the names of its
members — a hash of the sorted member list — never by its position in a tree. Two
cohorts can only add counts for "the same cluster" if that cluster is able to have
the same name in both. If each cohort clustered its own 80 most variable proteins,
the two sets would overlap only partly and most clusters could never be compared.

So the cohorts agree on a vocabulary first, and everything downstream uses it.

In [ ]:
import os, subprocess
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Every notebook reads and writes here, so the chain hands work along. In a real
# deployment this directory would be on a different machine for each cohort, and
# only the files named 'what this writes' would ever cross the boundary.
WORK = Path('sle-run'); WORK.mkdir(exist_ok=True); os.chdir(WORK)

def run(cmd):
    """Run one pvclust-py command and echo it, so the notebook shows the command
    line rather than hiding it behind a function."""
    print('$ ' + ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1800:])
    if r.returncode:
        raise SystemExit(f'command failed: {cmd}')

def show(png):
    """Render a written figure inline. matplotlib only, so this works in CI too."""
    if not Path(png).exists():
        print(f'(missing {png})'); return
    fig, ax = plt.subplots(figsize=(13, 13))
    ax.imshow(plt.imread(png)); ax.axis('off'); ax.set_title(png, fontsize=9)
    plt.show()

In [ ]:
run(['pvclust-py', 'shared-features',
     '--features', 'cohort-A_features.csv',
     '--features', 'cohort-B_features.csv'])

## How much was lost to the intersection?

Each cohort ranked its own 80 most variable proteins. The proteins that vary most
in a 262-sample cohort are not identical to those in a 94-sample cohort, so the
intersection is smaller than either list.

**If the intersection is very small, stop and look.** It means the two cohorts
disagree about what varies, which is itself a finding — different populations,
different assay behaviour — and federating over a handful of proteins will not
tell you much.

In [ ]:
a = pd.read_csv('cohort-A_features.csv')['feature']
b = pd.read_csv('cohort-B_features.csv')['feature']
shared = pd.read_csv('shared_features.csv')['feature']
print(f'cohort A ranked : {len(a)}')
print(f'cohort B ranked : {len(b)}')
print(f'shared          : {len(shared)}  ({100*len(shared)/min(len(a), len(b)):.0f}% of the smaller list)')
print()
print('in A only:', ', '.join(sorted(set(a) - set(b))[:6]) or 'none')
print('in B only:', ', '.join(sorted(set(b) - set(a))[:6]) or 'none')

---
**Next:** `04` and `05`. Each cohort clusters this shared vocabulary and ships its
sufficient statistics.